## Example 1: lark functionality test 1

In [ ]:
from lark import Lark

# Example 1: Basic parsing with simple arithmetic grammar
print("Example 1: Basic Parsing with Simple Arithmetic Grammar")

try:
    # Define a simple arithmetic grammar
    grammar = """
        start: expr
        
        ?expr: term
             | expr "+" term   -> add
             | expr "-" term   -> sub
             
        ?term: factor
             | term "*" factor -> mul
             | term "/" factor -> div
             
        ?factor: NUMBER
               | "(" expr ")"
               
        NUMBER: /[0-9]+/
        
        %import common.WS
        %ignore WS
    """
    
    # Create parser
    parser = Lark(grammar, start='start')
    print("Arithmetic parser created successfully")
    
    # Parse expressions
    expressions = [
        "42",
        "3 + 5",
        "10 - 2 * 3", 
        "(4 + 6) * 2",
        "100 / 5 + 7"
    ]
    
    print("Parsing arithmetic expressions:")
    for expr in expressions:
        try:
            tree = parser.parse(expr)
            print(f"  '{expr}' -> {tree.pretty()[:50].replace(chr(10), ' ')}")
        except Exception as e:
            print(f"  '{expr}' -> Parse error: {e}")
            raise
    
    # Show tree structure for complex expression
    complex_expr = "(10 + 5) * 2"
    tree = parser.parse(complex_expr)
    print(f"\nDetailed parse tree for '{complex_expr}':")
    print(tree.pretty())
    
    print("Basic parsing completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    # Fallback demonstration
    print("Demonstrating parsing concepts:")
    concepts = [
        "Context-free grammars: Define language syntax with production rules",
        "EBNF syntax: Extended Backus-Naur Form for grammar definition",
        "Parse trees: Hierarchical representation of parsed input",
        "Left recursion: Handles operator precedence and associativity",
        "Token transformation: Convert strings to structured data"
    ]
    
    for concept in concepts:
        print(f"  - {concept}")
    raise

## Example 2: lark functionality test 2

In [ ]:
from lark import Lark, Transformer

# Example 2: Grammar transformation and evaluation
print("Example 2: Grammar Transformation and Evaluation")

try:
    # Create a calculator transformer
    class Calculator(Transformer):
        def add(self, args):
            return args[0] + args[1]
        
        def sub(self, args):
            return args[0] - args[1]
            
        def mul(self, args):
            return args[0] * args[1]
            
        def div(self, args):
            return args[0] / args[1]
            
        def NUMBER(self, token):
            return int(token)
    
    # Grammar with transformation support
    calc_grammar = """
        ?start: expr
        
        ?expr: term
             | expr "+" term   -> add
             | expr "-" term   -> sub
             
        ?term: factor
             | term "*" factor -> mul
             | term "/" factor -> div
             
        ?factor: NUMBER
               | "(" expr ")"
               
        NUMBER: /[0-9]+/
        
        %import common.WS
        %ignore WS
    """
    
    # Create parser with LALR algorithm (supports transformers)
    parser = Lark(calc_grammar, parser='lalr')
    transformer = Calculator()
    print("Calculator parser with LALR algorithm created")
    
    # Evaluate expressions using separate transformation
    test_expressions = [
        "42",
        "7 + 3", 
        "15 - 8",
        "6 * 7",
        "20 / 4",
        "(3 + 4) * 5",
        "100 / (10 - 5)"
    ]
    
    print("Evaluating expressions with separate transformer:")
    for expr in test_expressions:
        try:
            tree = parser.parse(expr)
            result = transformer.transform(tree)
            print(f"  {expr} = {result}")
        except Exception as e:
            print(f"  {expr} -> Error: {e}")
            raise
    
    # Show transformer workflow
    print(f"\nTransformer workflow:")
    print(f"  1. Parse input into tree structure with LALR")
    print(f"  2. Apply transformer separately to parse tree")
    print(f"  3. Transform tree nodes using defined methods")
    print(f"  4. Return computed result instead of parse tree")
    
    # Complex expression evaluation
    complex_calc = "((10 + 5) * 3) / (20 - 15)"
    tree = parser.parse(complex_calc)
    result = transformer.transform(tree)
    print(f"\nComplex evaluation: {complex_calc} = {result}")
    
    print("Grammar transformation completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    # Fallback demonstration
    print("Demonstrating transformation concepts:")
    concepts = [
        "AST transformation: Convert parse trees to meaningful data",
        "Semantic actions: Execute code during parsing process",
        "Visitor pattern: Traverse and modify parse tree nodes",
        "Expression evaluation: Calculate results from parsed input",
        "Custom transformers: Implement domain-specific logic"
    ]
    
    for concept in concepts:
        print(f"  - {concept}")
    raise

## Example 3: lark functionality test 3

In [ ]:
from lark import Lark
from lark.exceptions import UnexpectedInput

# Example 3: JSON parser implementation
print("Example 3: JSON Parser Implementation")

try:
    # Define JSON grammar
    json_grammar = """
        ?value: object
              | array
              | string
              | SIGNED_NUMBER -> number
              | "true"        -> true
              | "false"       -> false
              | "null"        -> null

        object: "{" [pair ("," pair)*] "}"
        pair: string ":" value

        array: "[" [value ("," value)*] "]"

        string: ESCAPED_STRING

        %import common.ESCAPED_STRING
        %import common.SIGNED_NUMBER
        %import common.WS
        %ignore WS
    """
    
    # Create JSON parser
    json_parser = Lark(json_grammar, start='value')
    print("JSON parser created successfully")
    
    # Test JSON parsing
    json_examples = [
        '{"name": "John", "age": 30}',
        '[1, 2, 3, 4, 5]',
        '{"users": [{"id": 1, "active": true}, {"id": 2, "active": false}]}',
        '{"data": null, "count": 0}',
        '["hello", "world", "test"]'
    ]
    
    print("Parsing JSON examples:")
    for json_str in json_examples:
        try:
            tree = json_parser.parse(json_str)
            print(f"  ✓ Valid JSON: {json_str[:30]}...")
            # Show tree structure for first example
            if json_str == json_examples[0]:
                print(f"    Tree structure:\n{tree.pretty()}")
        except Exception as e:
            print(f"  ✗ Invalid JSON: {json_str[:30]}... -> {e}")
            raise
    
    # Test invalid JSON
    invalid_json = [
        '{"name": John}',  # Missing quotes
        '[1, 2, 3,]',     # Trailing comma
        '{"key": }',      # Missing value
    ]
    
    print("\nTesting invalid JSON (should fail):")
    for invalid in invalid_json:
        try:
            tree = json_parser.parse(invalid)
        except UnexpectedInput as e:
            print(f"  ✓ Correctly rejected: {invalid[:20]}... ({type(e).__name__} at column {e.column})")
        else:
            raise AssertionError(f"invalid JSON was accepted: {invalid}")
    
    # Show parser capabilities
    print(f"\nParser capabilities:")
    print(f"  • Recursive grammar rules for nested structures")
    print(f"  • Built-in terminal handling (strings, numbers)")
    print(f"  • Error detection for malformed input")
    print(f"  • Flexible grammar definition with EBNF")
    
    print("JSON parser implementation completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    # Fallback demonstration
    print("Demonstrating JSON parsing concepts:")
    concepts = [
        "Recursive rules: Handle nested objects and arrays",
        "Terminal imports: Reuse common patterns (strings, numbers)",
        "Grammar composition: Build complex parsers from simple rules", 
        "Error detection: Identify and report syntax violations",
        "Language implementation: Create parsers for existing formats"
    ]
    
    for concept in concepts:
        print(f"  - {concept}")
    raise

## Example 4: lark functionality test 4

In [ ]:
from lark import Lark, Visitor

# Example 4: Configuration file parser with visitor pattern
print("Example 4: Configuration File Parser with Visitor Pattern")

try:
    # Define configuration file grammar
    config_grammar = """
        start: (section | setting)*
        
        section: "[" section_name "]" setting*
        section_name: WORD
        
        setting: key "=" value
        key: WORD
        value: STRING | NUMBER | BOOLEAN
        
        STRING: /\\"[^\\"]*\\"/
        NUMBER: /[0-9]+/
        BOOLEAN: "true" | "false"
        WORD: /[a-zA-Z_][a-zA-Z0-9_]*/
        
        %import common.WS
        %ignore WS
    """
    
    # Create visitor to collect configuration data
    class ConfigVisitor(Visitor):
        def __init__(self):
            self.config = {}
            self.current_section = None
        
        def section_name(self, tree):
            self.current_section = tree.children[0].value
            if self.current_section not in self.config:
                self.config[self.current_section] = {}
        
        def setting(self, tree):
            key = tree.children[0].children[0].value
            value_node = tree.children[1].children[0]
            
            # Process different value types
            if value_node.type == 'STRING':
                value = value_node.value.strip('"')
            elif value_node.type == 'NUMBER':
                value = int(value_node.value)
            elif value_node.type == 'BOOLEAN':
                value = value_node.value == 'true'
            else:
                value = value_node.value
            
            # Store in current section or global
            if self.current_section:
                self.config[self.current_section][key] = value
            else:
                self.config[key] = value
    
    # Create parser
    config_parser = Lark(config_grammar)
    print("Configuration parser created successfully")
    
    # Sample configuration file content
    config_content = """
        debug = true
        port = 8080
        host = "localhost"
        
        [database]
        driver = "postgresql"
        port = 5432
        timeout = 30
        
        [logging]
        level = "info"
        file = "app.log"
        rotate = true
    """
    
    print(f"Parsing configuration content:")
    print(config_content.strip())
    
    # Parse and visit the tree
    tree = config_parser.parse(config_content)
    visitor = ConfigVisitor()
    visitor.visit(tree)
    
    print(f"\nExtracted configuration:")
    
    # Display parsed configuration
    for key, value in visitor.config.items():
        if isinstance(value, dict):
            print(f"  [{key}]:")
            for sub_key, sub_value in value.items():
                print(f"    {sub_key} = {sub_value} ({type(sub_value).__name__})")
        else:
            print(f"  {key} = {value} ({type(value).__name__})")
    
    # Show visitor pattern benefits
    print(f"\nVisitor pattern benefits:")
    print(f"  • Clean separation of parsing and data extraction")
    print(f"  • Multiple passes over the same parse tree")
    print(f"  • Type-specific processing of different nodes")
    print(f"  • Stateful processing with context tracking")
    
    print("Configuration parser with visitor completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    # Fallback demonstration
    print("Demonstrating visitor pattern concepts:")
    concepts = [
        "Tree traversal: Systematic visiting of parse tree nodes",
        "Stateful processing: Maintain context during tree walks",
        "Type dispatch: Handle different node types appropriately",
        "Data extraction: Convert parse trees to application data",
        "Multiple passes: Process same tree for different purposes"
    ]
    
    for concept in concepts:
        print(f"  - {concept}")
    raise

## Example 5: lark functionality test 5

In [ ]:
from lark import Lark

# Example 5: Advanced parsing features and grammar optimization
print("Example 5: Advanced Parsing Features and Grammar Optimization")

try:
    # Define grammar with advanced features
    advanced_grammar = """
        start: statement+
        
        statement: assignment
                 | function_call
                 | if_statement
        
        assignment: IDENTIFIER "=" expression ";"
        
        function_call: IDENTIFIER "(" [expression ("," expression)*] ")" ";"
        
        if_statement: "if" "(" condition ")" "{" statement* "}"
        
        condition: expression ">" expression -> gt
                 | expression "<" expression -> lt
        
        ?expression: term
                   | expression "+" term -> add
                   | expression "-" term -> sub
        
        ?term: factor
             | term "*" factor -> mul
             | term "/" factor -> div
             
        ?factor: IDENTIFIER
               | NUMBER
               | "(" expression ")"
        
        COMMENT: "//" /[^\\n]*/
        
        IDENTIFIER: /[a-zA-Z_][a-zA-Z0-9_]*/
        NUMBER: /[0-9]+/
        
        %import common.WS
        %ignore WS
        %ignore COMMENT
    """
    
    # Create parser with different parsing algorithms
    parsers = {}
    algorithms = ['earley', 'lalr']
    
    for algo in algorithms:
        try:
            parser = Lark(advanced_grammar, parser=algo)
            parsers[algo] = parser
            print(f"{algo.upper()} parser created successfully")
        except Exception as e:
            print(f"{algo.upper()} parser failed: {e}")
            raise
    
    # Sample code to parse
    code_sample = """
        // This is a comment
        x = 10 + 5;
        y = x * 2;
        print(y);
        if (x > 0) {
            result = x + y;
            display(result);
        }
    """
    
    print(f"\nParsing sample code:")
    print(code_sample.strip())
    
    # Test parsing with available algorithms
    assert set(parsers) == {'earley', 'lalr'}, parsers
    for algo, parser in parsers.items():
        try:
            tree = parser.parse(code_sample)
            print(f"\n{algo.upper()} parsing result:")
            print(f"  ✓ Successfully parsed {len(tree.children)} statements")
            
            # Show first statement tree
            if tree.children:
                first_stmt = tree.children[0]
                print(f"  First statement type: {first_stmt.data}")
            
            # The comment is ignored; x=, y=, print(...) and the if-block are the four statements.
            kinds = [stmt.children[0].data for stmt in tree.children]
            assert kinds == ['assignment', 'assignment', 'function_call', 'if_statement'], kinds
                
        except Exception as e:
            print(f"\n{algo.upper()} parsing failed: {e}")
            raise
    
    # Grammar optimization techniques
    print(f"\nGrammar optimization features:")
    optimization_features = [
        "Inline rules (?rule): Eliminate intermediate tree nodes",
        "Algorithm selection: Choose Earley for ambiguous, LALR for speed", 
        "Terminal priorities: Control token precedence",
        "Import optimization: Reuse common patterns",
        "Ignore directives: Skip whitespace and comments automatically"
    ]
    
    for feature in optimization_features:
        print(f"  • {feature}")
    
    # Show parsing performance considerations
    print(f"\nParsing performance tips:")
    performance_tips = [
        "Use LALR for unambiguous grammars (faster)",
        "Use Earley for ambiguous grammars (more flexible)",
        "Inline simple rules to reduce tree depth",
        "Import common terminals to avoid duplication",
        "Use ignore directive for non-semantic tokens"
    ]
    
    for tip in performance_tips:
        print(f"  - {tip}")
    
    print("Advanced parsing features completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    # Fallback demonstration
    print("Demonstrating advanced parsing concepts:")
    concepts = [
        "Algorithm selection: Different parsing strategies for different needs",
        "Grammar optimization: Techniques to improve parsing performance",
        "Ambiguity handling: Multiple parse trees for ambiguous input",
        "Parse tree control: Inline rules and tree transformation",
        "Language design: Best practices for grammar construction"
    ]
    
    for concept in concepts:
        print(f"  - {concept}")
    raise